<a href="https://colab.research.google.com/github/zerogradlabs/zerograd-python/blob/main/examples/evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluate and decide: honest splits, the right threshold, the columns that matter

A championship tells you which model wins on one holdout. Before a model goes to production you want three more answers:

1. **Does it hold on other splits?** Above all when rows repeat per customer or the model will predict the future.
2. **Where should the score cut?** 0.5 is rarely what the business wants.
3. **Which columns carry it?**

This notebook answers all three on a loan book with the open source SDK, locally: no account, no API key. At the end, the same models inside scikit-learn.

In [1]:
!pip install -q zerograd

## The data

A synthetic loan book: 600 customers with several loans each over two years. Default depends on the debt ratio, income and the
customer's own history (some customers are simply riskier), and the risk drifts up over time. Those two traits are what a random split hides.

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n_customers, n = 600, 4000
customer = rng.integers(0, n_customers, n)
risk = rng.normal(0, 1, n_customers)[customer]                     # each customer's own risk
date = pd.Timestamp("2024-01-01") + pd.to_timedelta(rng.integers(0, 730, n), unit="D")
months = (date - date.min()).days / 30
ratio = rng.beta(2, 5, n)
income = rng.lognormal(10.5, 0.5, n).round(-2)
purpose = rng.choice(["car", "home", "studies", "other"], n, p=[0.3, 0.3, 0.1, 0.3])
logit = -3 + 4 * ratio - 0.6 * (np.log(income) - 10.5) + 0.9 * risk + 0.03 * months + 0.5 * (purpose == "other")
loans = pd.DataFrame({
    "customer_id": customer, "date": date, "debt_ratio": ratio.round(3), "income": income, "purpose": purpose,
    "default": (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int),
})
loans.head()

,customer_id,date,debt_ratio,income,purpose,default
0,510,2024-01-24,0.413,34800.0,studies,1
1,382,2025-12-28,0.188,66400.0,car,1
2,306,2024-11-11,0.569,34700.0,car,1
3,161,2024-09-06,0.166,33300.0,car,0
4,184,2024-11-24,0.022,63000.0,other,0


In [3]:
loans["default"].mean().round(3), loans["customer_id"].value_counts().mean().round(1)

(np.float64(0.257), np.float64(6.7))

## 1. Cross-validation, three ways

`zg.cross_validate` runs every installed model on each fold and scores a baseline next to them. Binary tasks also get KS and lift in the top 10%.

In [4]:
from zerograd import ZeroGrad

zg = ZeroGrad(models=["tabpfn-v2", "tabicl-v2"])   # leave models out to run every installed model
data = loans.drop(columns=["customer_id", "date"])
random_cv = zg.cross_validate(data, "default", folds=5)
random_cv.summary

/home/runner/work/zerograd-python/zerograd-python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
◆ ZeroGrad 0.2.4 · local · 2 models installed: TabPFN v2, TabICLv2 · weights download on first use


  Hold on tight, going gradless.


↓ TabPFN v2: fetching weights from huggingface.co/Prior-Labs/TabPFN-v2-clf (first time only) · Downloading a few million synthetic datasets' worth of intuition.


↓ TabICLv2: fetching weights from huggingface.co/jingang/TabICL (first time only) · Unpacking weights. You'll only wait for this once.


◆ Cross-validation · 5 folds stratified · 2 models · 4,000 rows


… Fold 1 of 5


... (8 outputs left out)

  🥇 tabicl-v2  ████████████████████████ 0.666


  🥈 tabpfn-v2  ████ 0.664


◆ Champion: tabicl-v2 · mean ROC AUC 0.666


,roc_auc,roc_auc_std,ks,lift_at_10,accuracy,balanced_accuracy,log_loss,pr_auc,f1,precision,recall,folds,seconds
model,,,,,,,,,,,,,
tabicl-v2,0.665502,0.021157,0.274059,1.865925,0.74125,0.516465,0.539793,0.400581,0.095913,0.479931,0.053460,5,172.04
tabpfn-v2,0.663762,0.021575,0.272220,1.856216,0.74175,0.517114,0.540469,0.400196,0.097179,0.496374,0.054421,5,267.68
baseline:majority,0.500000,0.000000,0.000000,1.000000,0.74275,0.500000,0.570161,0.257250,0.000000,0.000000,0.000000,5,0.00


Now split **by customer**: every loan of a customer lands on the same side, so a model is never scored on someone it already read in its
context. This is the number to trust when the model will score new customers.

In [5]:
by_customer = zg.cross_validate(loans.drop(columns=["date"]), "default", folds=5, groups="customer_id")
by_customer.summary

◆ Cross-validation · 5 folds by group · 2 models · 4,000 rows


… Fold 1 of 5


✓ Fold 1 of 5


… Fold 2 of 5


✓ Fold 2 of 5


… Fold 3 of 5


... (4 outputs left out)

  🥇 tabicl-v2  ████████████████████████ 0.664


  🥈 tabpfn-v2  ████ 0.663


◆ Champion: tabicl-v2 · mean ROC AUC 0.664


,roc_auc,roc_auc_std,ks,lift_at_10,accuracy,balanced_accuracy,log_loss,pr_auc,f1,precision,recall,folds,seconds
model,,,,,,,,,,,,,
tabicl-v2,0.663676,0.020271,0.273625,1.836878,0.741500,0.516287,0.540656,0.397353,0.094260,0.473929,0.052435,5,172.76
tabpfn-v2,0.663126,0.022038,0.273903,1.826937,0.743001,0.516987,0.540685,0.397912,0.093376,0.506306,0.051473,5,263.55
baseline:majority,0.500000,0.000000,0.000000,1.000000,0.742750,0.500000,0.570165,0.257250,0.000000,0.000000,0.000000,5,0.00


And **by time**: train on the past, score the next block, walk forward. This is the number to trust when the model will score next
month's applications. `gap=` leaves rows out between the two sides, for targets that take a while to be known.

In [6]:
by_time = zg.cross_validate(loans.drop(columns=["customer_id"]), "default", folds=4, time="date")
by_time.summary

◆ Cross-validation · 4 folds by time · 2 models · 4,000 rows


… Fold 1 of 4


✓ Fold 1 of 4


… Fold 2 of 4


✓ Fold 2 of 4


… Fold 3 of 4


... (2 outputs left out)

  🥇 tabicl-v2  ████████████████████████ 0.668


  🥈 tabpfn-v2  ████ 0.668


◆ Champion: tabicl-v2 · mean ROC AUC 0.668


,roc_auc,roc_auc_std,ks,lift_at_10,accuracy,balanced_accuracy,log_loss,pr_auc,f1,precision,recall,folds,seconds
model,,,,,,,,,,,,,
tabicl-v2,0.668321,0.020031,0.275501,1.810935,0.729688,0.505420,0.562003,0.420408,0.031972,0.383242,0.016902,4,80.45
tabpfn-v2,0.668115,0.021745,0.276296,1.802282,0.729688,0.506119,0.560785,0.422862,0.035882,0.497059,0.019175,4,115.16
baseline:majority,0.500000,0.000000,0.000000,1.000000,0.729375,0.500000,0.588222,0.270625,0.000000,0.000000,0.000000,4,0.00


In [7]:
pd.DataFrame({
    "random": random_cv.summary["roc_auc"],
    "by customer": by_customer.summary["roc_auc"],
    "by time": by_time.summary["roc_auc"],
}).round(3)

,random,by customer,by time
model,,,
tabicl-v2,0.666,0.664,0.668
tabpfn-v2,0.664,0.663,0.668
baseline:majority,0.500,0.500,0.500


`.folds` has one row per model and fold, if you want the spread behind the mean.

In [8]:
by_time.folds[["model", "fold", "train_rows", "test_rows", "roc_auc", "ks", "lift_at_10"]].round(3)

,model,fold,train_rows,test_rows,roc_auc,ks,lift_at_10
0,tabpfn-v2,0,800,800,0.674,0.292,1.502
1,tabicl-v2,0,800,800,0.677,0.288,1.596
2,baseline:majority,0,800,800,0.500,0.000,1.000
3,tabpfn-v2,1,1600,800,0.696,0.283,2.029
4,tabicl-v2,1,1600,800,0.693,0.287,1.836
5,baseline:majority,1,1600,800,0.500,0.000,1.000
6,tabpfn-v2,2,2400,800,0.649,0.261,1.814
7,tabicl-v2,2,2400,800,0.649,0.266,1.903
8,baseline:majority,2,2400,800,0.500,0.000,1.000
9,tabpfn-v2,3,3200,800,0.654,0.270,1.864


## 2. Pick the threshold

Train a capability on everything but the last quarter, then pick the cut. Say a missed default costs 20 times what a false alarm costs (a
manual review). `optimize_threshold` reads the confusion counts the championship saved for every cut, so it needs no new predictions.

In [9]:
cut = loans["date"] < "2025-10-01"
train, recent = loans[cut].drop(columns=["customer_id", "date"]), loans[~cut].drop(columns=["customer_id", "date"])
cap = zg.create("loan-default", examples=train, target="default")

✓ 3,526 labeled rows · 3 columns used


◆ Championship · 2 models · Let them read the room.


✓ Context 2,821 rows · holdout 705 rows (every row scored)


… TabPFN v2 (1 of 2)


✓ TabPFN v2 (1 of 2) 42.9s


… TabICLv2 (2 of 2)


... (2 outputs left out)

  🥈 TabICLv2   ███████████████████████ 0.667


     without a model  0.500  (ROC AUC, majority class)


◆ Champion: TabPFN v2 · ROC AUC 0.678 · Built with PriorLabs-TabPFN


  Gradient-free and proud of it.


In [10]:
by_cost = cap.optimize_threshold("cost", cost_fp=1, cost_fn=20)
by_cost

<ThresholdResult · cost · threshold 0.11 · precision 0.276 · recall 1.000 · flags 92.5% of rows · from the championship holdout (705 rows)>

In [11]:
by_cost.table.loc[[0.1, 0.2, 0.3, 0.5]].round(3)

,tp,fp,fn,tn,precision,recall,f1,fpr,cost,flagged
threshold,,,,,,,,,,
0.1,180.0,503.0,0.0,22.0,0.264,1.000,0.417,0.958,503.0,0.969
0.2,129.0,230.0,51.0,295.0,0.359,0.717,0.479,0.438,1250.0,0.509
0.3,78.0,109.0,102.0,416.0,0.417,0.433,0.425,0.208,2149.0,0.265
0.5,8.0,6.0,172.0,519.0,0.571,0.044,0.082,0.011,3446.0,0.020


Other objectives: the most recall while 8 in 10 flags are right, the best F1, or Youden's J. `apply=False` only looks.

In [12]:
for objective, kw in [("recall", {"min_precision": 0.5}), ("f1", {}), ("youden", {})]:
    try:
        print(cap.optimize_threshold(objective, apply=False, **kw))
    except Exception as e:  # a constraint no cut meets says so
        print(objective, "->", e)

<ThresholdResult · recall · threshold 0.42 · precision 0.518 · recall 0.161 · flags 7.9% of rows · from the championship holdout (705 rows)>
<ThresholdResult · f1 · threshold 0.20 · precision 0.359 · recall 0.717 · flags 50.9% of rows · from the championship holdout (705 rows)>
<ThresholdResult · youden · threshold 0.20 · precision 0.359 · recall 0.717 · flags 50.9% of rows · from the championship holdout (705 rows)>


The capability now predicts with the cost-optimal cut. Check it on the quarter it never saw:

In [13]:
from sklearn.metrics import confusion_matrix

preds = cap.predict(recent.drop(columns="default"))
print("threshold", cap.threshold)
confusion_matrix(recent["default"], preds["prediction"])

… Warming up TabPFN v2 with 3,526 rows of context · In-context learning, in your context.


✓ TabPFN v2 ready 50.2s


threshold 0.11


array([[ 40, 306],
       [  5, 123]])

## 3. Which columns matter

Shuffle one column at a time in rows the model did not read, and see how much ROC AUC drops.

In [14]:
cap.importance(recent, repeats=3)

… Importance · 3 columns × 3 · 474 rows


✓ Importance of 3 columns 1m06s


,importance,std
column,,
debt_ratio,0.149687,0.023622
income,0.007508,0.011053
purpose,0.006187,0.000148


## 4. Inside scikit-learn

`ZeroGradClassifier` and `ZeroGradRegressor` are scikit-learn estimators: `fit` keeps a context, `predict_proba` runs the model with it. They
work in pipelines, grid searches and every scikit-learn tool that takes an estimator. You pick the model; there is no championship inside.

In [15]:
from sklearn.model_selection import GroupKFold, cross_val_score

from zerograd import ZeroGradClassifier

X, y = loans[["debt_ratio", "income", "purpose"]], loans["default"]
clf = ZeroGradClassifier(model="tabicl-v2")
cross_val_score(clf, X, y, groups=loans["customer_id"], cv=GroupKFold(5), scoring="roc_auc").round(3)

array([0.632, 0.676, 0.635, 0.646, 0.702])

In [16]:
from sklearn.inspection import permutation_importance

clf.fit(X[cut], y[cut])
r = permutation_importance(clf, X[~cut], y[~cut], scoring="roc_auc", n_repeats=3, random_state=0)
pd.Series(r.importances_mean, index=X.columns).sort_values(ascending=False).round(4)

debt_ratio    0.1362
purpose       0.0093
income        0.0084
dtype: float64

## Next

* `zg.context_curve(...)` finds the smallest context that scores as well, for faster predictions.
* The [no target notebook](https://github.com/zerogradlabs/zerograd-python/blob/main/examples/no-target.ipynb) covers anomalies, imputation, synthetic rows and clustering.
* Docs: [zerograd.dev/docs#evaluation](https://zerograd.dev/docs#evaluation).